# Planning and Orchestration

Planning turns an agent from a single response into a multi-step workflow: the model decides which step comes next. That autonomy needs control. Useful orchestration is inspectable and bounded, with enough autonomy to complete the task and no more.

In this notebook, you limit how much the operations assistant may do, verify its finished trace against trusted ticket data, and see why recording that a review is needed is not the same as enforcing one.

## Learning Objectives

- Bound model requests and tool calls with `UsageLimits`.
- Verify routing and review requirements against trusted ticket data.
- Distinguish a pending-review marker from enforced human approval.

## Mental Model

An agent that plans runs a loop: the model picks the next step, the application runs it, and the result feeds the next choice. The application controls that loop at three points:

- **During the run:** usage limits cap model requests and tool calls, so a confused agent cannot loop forever.
- **After the run:** a verification step checks the finished trace against trusted data, not against the model's own claims.
- **For risky decisions:** a human review. Here the agent only records that a review is needed; a real gate would pause the run until a person approves.


```mermaid
stateDiagram-v2
    [*] --> Received
    Received --> TicketLoaded: lookup_ticket
    Received --> Failed: no ticket ID
    TicketLoaded --> EvidenceGathered: lookup_policy + find_available_owner + recall_memory
    EvidenceGathered --> ReviewRecorded: priority == critical
    EvidenceGathered --> Proposed: otherwise
    ReviewRecorded --> Proposed: human_review returns approved = False
    Proposed --> Checked: verify_trace against trusted ticket
    Received --> Stopped: UsageLimits exceeded
    TicketLoaded --> Stopped: UsageLimits exceeded
    EvidenceGathered --> Stopped: UsageLimits exceeded
    Checked --> [*]
    Stopped --> [*]
    Failed --> [*]
```

Planning is often drawn as a decision cycle: propose candidate actions, evaluate them, select one, execute it, then observe the result and plan again. Here, the configured model proposes tool calls. `UsageLimits` bounds the running loop; `verify_trace` checks the finished proposal against the ticket data.

<div align="center">
  <img src="assets/03-coala-decision-cycle.png" alt="CoALA decision cycle: proposal, evaluation, selection, execution" width="260" style="max-width: 100%; height: auto;" />
  <p><em>The CoALA decision cycle: planning proposes, evaluates, and selects an action, execution acts on the environment, and the next observation starts a new cycle. Source: <a href="https://arxiv.org/abs/2309.02427" target="_blank" rel="noopener noreferrer">Sumers et al., 2023</a>, CC BY 4.0</em></p>
</div>

The flowchart shows the routing rules for one ticket. The workflow only returns a routing proposal: nothing here executes an incident response or sends a customer update.

```mermaid
flowchart TD
    A["Ticket request"] --> B{"Ticket ID in request<br/>or message history?"}
    B -- no --> X["Reject: ticket ID required"]
    B -- yes --> C["lookup_ticket"]
    C --> D["lookup_policy(topic)"]
    D --> E["find_available_owner(topic, region)"]
    E --> F{"priority == critical?"}
    F -- yes --> G["human_review: record pending status,<br/>not approval"]
    F -- no --> H["Automated routing"]
    G --> I["AgentDecision<br/>requires_human_review = true"]
    H --> J["AgentDecision<br/>requires_human_review = false"]
```

## Load The Shared Helpers


In [ ]:
from pydantic_ai.exceptions import UsageLimitExceeded
from pydantic_ai.usage import UsageLimits

from support import (
    agent_limits,
    build_agent,
    load_operations_data,
    trace_from_result,
    verify_trace,
)

## Run A Critical Incident Within Limits

Usage limits are a practical control against runaway loops. They cap how much the agent can do before the application stops and asks for intervention. A request is one call to the model; a tool call is one tool the application executes for it. When a run would exceed either limit, PydanticAI raises `UsageLimitExceeded` and the run ends without a decision.


In [ ]:
# The scripted critical path uses four requests and five tools; live paths may vary.
limits = agent_limits()
data = load_operations_data()
agent = build_agent(data)
print("Model:", type(agent.model).__name__)
limits

The selected model proposes tool calls, while PydanticAI validates and executes them. Critical tickets must call `human_review`, which marks approval as pending. It never approves or sends a customer update. Inspect `verify_trace` to detect a live model skipping that step; the prompt is not an enforcement guarantee.

In [ ]:
request = "Handle the critical export incident in T-1003."
critical_result = await agent.run(request, usage_limits=limits)
critical_trace = trace_from_result(critical_result, request, case_id="critical")
[(step.tool, step.arguments) for step in critical_trace.steps]

In [ ]:
critical_trace.decision.model_dump()

The decision says a review is required, but look at what `human_review` actually returned. The review is recorded as required and not approved, yet the run still finished and returned a decision: nothing waited for a person.

In [ ]:
[
    part.content
    for message in critical_result.new_messages()
    for part in message.parts
    if part.part_kind == "tool-return" and part.tool_name == "human_review"
]

Limits act while the agent runs. The scripted incident cannot finish within one request. A live model may also stop at that limit; if it returns a decision immediately, inspect its trace for missing tools rather than treating that shortcut as success:

In [ ]:
try:
    limited_result = await agent.run(request, usage_limits=UsageLimits(request_limit=1))
    print(limited_result.output.model_dump())
except UsageLimitExceeded as error:
    print(f"Stopped: {error}")

## Verify The Finished Trace

Usage limits only bound how much the agent does. A separate check decides whether what it did is acceptable. `verify_trace` reads the requested ticket from the trusted data and checks two things:

- `within_step_budget`: the trace has at most five tool steps.
- `human_review_gate`: the decision is for the requested ticket, its review flag matches the ticket's real priority, and a critical ticket has a `human_review` call for that same ticket (a non-critical one has none).

The check uses the ticket data, not the model's `requires_human_review` flag, so a wrong flag cannot pass by agreeing with itself.

In [ ]:
print(critical_result.usage)
verify_trace(critical_trace, data)

## Exercise: Write A Verification Check

`verify_trace` checks the review gate, but not whether the right person owns the ticket. Add a check of your own that uses trusted data:

- write `owner_in_region(trace, data)` that returns `True` when the decision's `owner` appears in `data.availability` for the same region as the decision's ticket,
- test it on `critical_trace`, which should pass,
- test it on a copy of `critical_trace` whose decision names `Priya` as the owner, which should fail. Build the copy with `dataclasses.replace` and the decision's `model_copy(update=...)`, without rerunning the agent,
- run `verify_trace` on the copy as well, and compare.

In [ ]:
# @TODO: Write owner_in_region, then test it on critical_trace and a wrong-owner copy.


<details>
<summary>Show Reference Solution</summary>

```python
from dataclasses import replace


def owner_in_region(trace, data):
    ticket = data.lookup_ticket(trace.decision.ticket_id)
    regional_owners = {
        row["owner"] for row in data.availability if row["region"] == ticket["region"]
    }
    return trace.decision.owner in regional_owners


wrong_owner_trace = replace(
    critical_trace,
    decision=critical_trace.decision.model_copy(update={"owner": "Priya"}),
)
print("Real trace:", owner_in_region(critical_trace, data))
print("Wrong owner:", owner_in_region(wrong_owner_trace, data))
print("verify_trace on wrong owner:", verify_trace(wrong_owner_trace, data))
```

</details>


The real trace passes: Sam works in APAC, the ticket's region. The copy names Priya, who works in the EU, so your check rejects it. `verify_trace` still passes the copy, because it never looks at the owner. A verification step only catches what it checks, so each rule that matters needs its own check against trusted data.

## Choose An Orchestration Pattern

Sequential plans are easiest to inspect. Parallel or multi-agent patterns become useful when independent subtasks can run safely without stepping on shared state.

| Pattern | Best for | Risk |
|---|---|---|
| **Single agent** | Small workflows with one decision owner | Model may try too many actions without limits |
| **Orchestrator-workers** | Independent subtasks with clear merge rules | Harder trace review and conflict resolution |
| **Crew** | Role-based collaboration demos or specialist tasks | Coordination overhead can exceed task value |

## Check Your Understanding

1. What is the difference between recording that a decision needs human review and enforcing that review, and when do you need enforcement?
2. Why is a limit that stops an agent while it runs stronger than a check on the finished trace, and why do you still need the check?
3. Why should a verification step read the ticket's priority from trusted data instead of trusting the model's review flag?
4. When does an orchestrator-workers pattern make sense instead of a single agent?

<details>
<summary>Show Reference Answers</summary>

1. Recording only marks the decision: the run continues and returns, as `human_review` did here with `approved` set to `False`. Enforcing means the run pauses until a person approves, before any action is taken. You need enforcement as soon as the agent can execute sensitive or irreversible actions, such as sending a customer update or starting an incident response.
2. A limit stops the loop while it runs, so the agent cannot make more model requests or tool calls than allowed. A check on the finished trace only sees what already happened. You still need the check because a run can stay within its limits and still produce a wrong decision, such as a missing review.
3. The model's output can be wrong, and a check that compares the model's flag with itself would always agree. Reading the priority from trusted data gives an independent source, so a missing or wrong review flag is caught.
4. When the work splits into independent subtasks with clear rules for merging their results. The cost is harder trace review and conflict resolution, so a single agent stays the better default for a small workflow with one decision owner.

</details>


## Further Reading

- [PydanticAI Usage Limits](https://pydantic.dev/docs/ai/core-concepts/agent/#usage-limits)
- [PydanticAI Human-In-The-Loop Tool Approval](https://pydantic.dev/docs/ai/tools-toolsets/deferred-tools/#human-in-the-loop-tool-approval)
